In [1]:
from google.colab import files
uploaded = files.upload()

Saving phishing.csv to phishing.csv


In [2]:
import pandas as pd

df = pd.read_csv('phishing.csv')
print(df.shape)
df.head()

(11054, 32)


,Index,UsingIP,LongURL,ShortURL,Symbol@,Redirecting//,PrefixSuffix-,SubDomains,HTTPS,DomainRegLen,...,UsingPopupWindow,IframeRedirection,AgeofDomain,DNSRecording,WebsiteTraffic,PageRank,GoogleIndex,LinksPointingToPage,StatsReport,class
0,0,1,1,1,1,1,-1,0,1,-1,...,1,1,-1,-1,0,-1,1,1,1,-1
1,1,1,0,1,1,1,-1,-1,-1,-1,...,1,1,1,-1,1,-1,1,0,-1,-1
2,2,1,0,1,1,1,-1,-1,-1,1,...,1,1,-1,-1,1,-1,1,-1,1,-1
3,3,1,0,-1,1,1,-1,1,1,-1,...,-1,1,-1,-1,0,-1,1,1,1,1
4,4,-1,0,-1,1,-1,-1,1,1,-1,...,1,1,1,1,1,-1,1,-1,-1,1


In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Drop the Index column - it's not useful information
df = df.drop('Index', axis=1)

# X = the clues, y = the answer (phishing or legitimate)
X = df.drop('class', axis=1)
y = df['class']

# Split data: 80% to teach the model, 20% to test it
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create and train the model
model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

# Test how well it learned
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print(f"Accuracy: {accuracy*100:.2f}%")

Accuracy: 96.97%


In [4]:
import pandas as pd

importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Top 10 most important clues for detecting phishing:")
print(importances.head(10))


Top 10 most important clues for detecting phishing:
HTTPS                  0.323783
AnchorURL              0.243877
WebsiteTraffic         0.074193
SubDomains             0.063427
LinksInScriptTags      0.042505
PrefixSuffix-          0.041890
LinksPointingToPage    0.020086
RequestURL             0.019931
ServerFormHandler      0.019764
DomainRegLen           0.016330
dtype: float64


In [5]:
import joblib

joblib.dump(model, 'phishing_model.pkl')
print("Model saved!")

Model saved!


In [6]:
# These are the clues we can compute instantly for ANY live URL
live_features = [
    'UsingIP', 'LongURL', 'ShortURL', 'Symbol@', 'Redirecting//',
    'PrefixSuffix-', 'SubDomains', 'HTTPS', 'Favicon', 'NonStdPort',
    'HTTPSDomainURL', 'RequestURL', 'AnchorURL', 'LinksInScriptTags',
    'ServerFormHandler', 'InfoEmail', 'WebsiteForwarding', 'StatusBarCust',
    'DisableRightClick', 'UsingPopupWindow', 'IframeRedirection'
]

X_live = df[live_features]
y_live = df['class']

X_train2, X_test2, y_train2, y_test2 = train_test_split(X_live, y_live, test_size=0.2, random_state=42)

live_model = RandomForestClassifier(n_estimators=200, random_state=42)
live_model.fit(X_train2, y_train2)

live_predictions = live_model.predict(X_test2)
live_accuracy = accuracy_score(y_test2, live_predictions)
print(f"Live-friendly model accuracy: {live_accuracy*100:.2f}%")

# Save this version too
joblib.dump(live_model, 'phishing_model_live.pkl')
print("Live model saved!")

Live-friendly model accuracy: 95.25%
Live model saved!


In [7]:
import re
import requests
from urllib.parse import urlparse
from bs4 import BeautifulSoup

SHORTENERS = ['bit.ly', 'tinyurl.com', 'goo.gl', 't.co', 'ow.ly', 'is.gd',
              'buff.ly', 'adf.ly', 'shorte.st', 'rebrand.ly']

def extract_features(url, timeout=6):
    if not url.startswith(('http://', 'https://')):
        url = 'http://' + url

    parsed = urlparse(url)
    domain = parsed.netloc
    features = {}

    features['UsingIP'] = -1 if re.match(r'^(\d{1,3}\.){3}\d{1,3}$', domain.split(':')[0]) else 1
    features['LongURL'] = -1 if len(url) >= 75 else (0 if len(url) >= 54 else 1)
    features['ShortURL'] = -1 if any(s in domain for s in SHORTENERS) else 1
    features['Symbol@'] = -1 if '@' in url else 1
    features['Redirecting//'] = -1 if url.rfind('//') > 7 else 1
    features['PrefixSuffix-'] = -1 if '-' in domain else 1
    dots = domain.split(':')[0].count('.')
    features['SubDomains'] = 1 if dots <= 1 else (0 if dots == 2 else -1)
    features['HTTPS'] = 1 if parsed.scheme == 'https' else -1
    features['NonStdPort'] = -1 if (':' in domain and domain.split(':')[1] not in ('80', '443')) else 1
    features['HTTPSDomainURL'] = -1 if 'https' in domain.lower() else 1

    try:
        resp = requests.get(url, timeout=timeout, headers={'User-Agent': 'Mozilla/5.0'}, allow_redirects=True)
        html = resp.text
        soup = BeautifulSoup(html, 'html.parser')
        final_domain = urlparse(resp.url).netloc

        icon = soup.find('link', rel=lambda x: x and 'icon' in x.lower())
        if icon and icon.get('href'):
            icon_domain = urlparse(icon['href']).netloc
            features['Favicon'] = 1 if (icon_domain == '' or final_domain in icon_domain) else -1
        else:
            features['Favicon'] = 1

        tags = soup.find_all(['img', 'script', 'iframe', 'link'])
        ext = sum(1 for t in tags if t.get('src') and final_domain not in t.get('src') and t.get('src').startswith('http'))
        total = max(len(tags), 1)
        ratio = ext / total
        features['RequestURL'] = 1 if ratio < 0.22 else (0 if ratio < 0.61 else -1)

        anchors = soup.find_all('a')
        bad_anchor = sum(1 for a in anchors if not a.get('href') or a.get('href').startswith('#') or 'javascript:void' in (a.get('href') or ''))
        ext_anchor = sum(1 for a in anchors if a.get('href', '').startswith('http') and final_domain not in a.get('href', ''))
        total_a = max(len(anchors), 1)
        anchor_ratio = (bad_anchor + ext_anchor) / total_a
        features['AnchorURL'] = 1 if anchor_ratio < 0.31 else (0 if anchor_ratio < 0.67 else -1)

        ls_tags = soup.find_all(['link', 'script'])
        ls_ext = sum(1 for t in ls_tags if (t.get('src') or t.get('href')) and final_domain not in (t.get('src') or t.get('href') or ''))
        ls_total = max(len(ls_tags), 1)
        features['LinksInScriptTags'] = 1 if (ls_ext / ls_total) < 0.17 else -1

        forms = soup.find_all('form')
        suspicious_form = any((f.get('action') in (None, '', 'about:blank')) or
                               (f.get('action', '').startswith('http') and final_domain not in f.get('action', ''))
                               for f in forms)
        features['ServerFormHandler'] = -1 if suspicious_form else 1
        features['InfoEmail'] = -1 if 'mailto:' in html.lower() else 1

        n_redirects = len(resp.history)
        features['WebsiteForwarding'] = 1 if n_redirects <= 1 else (0 if n_redirects <= 3 else -1)
        features['StatusBarCust'] = -1 if 'onmouseover' in html.lower() else 1
        features['DisableRightClick'] = -1 if 'event.button==2' in html.lower() or 'oncontextmenu' in html.lower() else 1
        features['UsingPopupWindow'] = -1 if 'window.open(' in html.lower() else 1
        features['IframeRedirection'] = -1 if soup.find('iframe') else 1

    except Exception:
        for k in ['Favicon', 'RequestURL', 'AnchorURL', 'LinksInScriptTags', 'ServerFormHandler',
                  'InfoEmail', 'WebsiteForwarding', 'StatusBarCust', 'DisableRightClick',
                  'UsingPopupWindow', 'IframeRedirection']:
            features[k] = 0

    return [features[k] for k in live_features]

print("Function ready!")

Function ready!


In [8]:
test_url = "https://www.google.com"

feats = extract_features(test_url)
prediction = live_model.predict([feats])[0]
proba = live_model.predict_proba([feats])[0]

result = "Legitimate ✅" if prediction == 1 else "Phishing ⚠️"
print(f"URL: {test_url}")
print(f"Result: {result}")
print(f"Confidence: {max(proba)*100:.1f}%")

URL: https://www.google.com
Result: Legitimate ✅
Confidence: 100.0%


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [9]:
test_url2 = "https://www.wikipedia.org"

feats2 = extract_features(test_url2)
prediction2 = live_model.predict([feats2])[0]
proba2 = live_model.predict_proba([feats2])[0]

result2 = "Legitimate ✅" if prediction2 == 1 else "Phishing ⚠️"
print(f"URL: {test_url2}")
print(f"Result: {result2}")
print(f"Confidence: {max(proba2)*100:.1f}%")

URL: https://www.wikipedia.org
Result: Legitimate ✅
Confidence: 100.0%


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [14]:
user_url = input("Enter a URL to check: ")

feats3 = extract_features(user_url)
prediction3 = live_model.predict([feats3])[0]
proba3 = live_model.predict_proba([feats3])[0]

result3 = "Legitimate ✅" if prediction3 == 1 else "Phishing ⚠️"
print(f"\nURL: {user_url}")
print(f"Result: {result3}")
print(f"Confidence: {max(proba3)*100:.1f}%")

Enter a URL to check: www.paypal-verify-account.com

URL: www.paypal-verify-account.com
Result: Phishing ⚠️
Confidence: 63.2%


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
